# LLamaFactory

使用llamafactory来进行0代码微调，需要配置的文件有两个:
1. dataset_info.json 文件 主要是用来注册数据集，注册好的数据直接放到lora_sft.yaml中使用
2. lora_sft.yaml 用来配置llamafactory里面的各种数据



In [6]:
!type dataset_info.json

/bin/bash: line 1: type: dataset_info.json: not found


anti_fraud_0819_train 和 anti_fraud_0819_eval 和 anti_fraud_0819_test 是注册好的数据集名字，放在lora_sft.yaml中使用

In [7]:
!type anti_fraud_qwen2_lora_sft.yaml

/bin/bash: line 1: type: anti_fraud_qwen2_lora_sft.yaml: not found


In [8]:
# AutoDL/Linux: 如果上传后当前目录不在第 11 章目录，先切换到这里
# 按你的实际上传路径修改这一行。例如：/root/autodl-tmp/StudyLLM/11.Anti-Frad
%cd /root/autodl-tmp/StudyLLM/11.Anti-Frad

/root/autodl-tmp/StudyLLM/11.Anti-Frad


## AutoDL 1：检查数据和配置文件

在 AutoDL 上是 Linux 环境，所以查看文件用 `cat`，不是 Windows 的 `type`。

In [9]:
!pwd
!ls -lh
!ls -lh data/train_test
!cat dataset_info.json
!cat anti_fraud_qwen2_lora_sft.yaml

/root/autodl-tmp/StudyLLM/11.Anti-Frad
total 568K
-rw-r--r-- 1 root root 134K May 24 19:23 '01. 大模型调用.ipynb'
-rw-r--r-- 1 root root 6.7K May 24 19:23 '02. 选择模型.ipynb'
-rw-r--r-- 1 root root 100K May 24 19:23 '03 制作数据集.ipynb'
-rw-r--r-- 1 root root 132K May 24 19:23 '04 train_eval_package.ipynb'
-rw-r--r-- 1 root root  85K May 24 19:23 '05. Lora单卡训练.ipynb'
-rw-r--r-- 1 root root  46K May 24 19:54 '06. LLamaFactory.ipynb'
-rw-r--r-- 1 root root  47K May 24 19:23  _orig_positive.ipynb
-rw-r--r-- 1 root root  786 May 24 19:57  anti_fraud_qwen2_lora_sft.yaml
-rw-r--r-- 1 root root  692 May 24 20:05  dataset_info.json
ls: cannot access 'data/train_test': No such file or directory
{
  "anti_fraud_0819_train": {
    "file_name": "/root/autodl-tmp/StudyLLM/datasets/11/train_test/train0819_alpaca.jsonl",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  },
  "anti_fraud_0819_eval": {
    "file_name": "/root/autodl-tmp/StudyLLM/datasets/11/

## AutoDL 2：下载并安装 LLaMA-Factory

这一步只需要在 AutoDL 的训练环境里执行一次。后面如果已经安装过，可以跳过。

In [10]:
%%bash
set -e
cd /root/autodl-tmp
if [ ! -d LLaMA-Factory ]; then
  git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
fi
cd LLaMA-Factory
pip install -e .
pip install -r requirements/metrics.txt
llamafactory-cli version || true

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Obtaining file:///root/autodl-tmp/LLaMA-Factory
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for llamafactory (pyproject.toml): started
  Building editable for llamafactory (pyproject.toml): finished with status 'done'
  Created wheel for llamafactory: filename=llamafactory-0.9.5.dev0-py3-none-any.whl size=27090 sha256=f08d4db00239fc69dc4d763d1cbecc544b37e261eb7a

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


ERROR: Operation cancelled by user


Process was interrupted.


CalledProcessError: Command 'b'set -e\ncd /root/autodl-tmp\nif [ ! -d LLaMA-Factory ]; then\n  git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git\nfi\ncd LLaMA-Factory\npip install -e .\npip install -r requirements/metrics.txt\nllamafactory-cli version || true\n'' returned non-zero exit status 1.

## 下载下要微调的模型

In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    "Qwen/Qwen2-0.5B-Instruct",
    cache_dir="/root/autodl-tmp/StudyLLM/models"
)

print(model_dir)

2026-05-24 19:55:15,507 - modelscope - INFO - Got 10 files, start to download ...


Processing 10 items:   0%|          | 0.00/10.0 [00:00<?, ?it/s]

2026-05-24 19:56:15,833 - modelscope - INFO - Finish downloading 10 files for repo 'Qwen/Qwen2-0.5B-Instruct'
2026-05-24 19:56:15,834 - modelscope - INFO - Creating symbolic link [/root/autodl-tmp/StudyLLM/models/Qwen/Qwen2-0.5B-Instruct].


/root/autodl-tmp/StudyLLM/models/Qwen/Qwen2-0___5B-Instruct


## AutoDL 3：确认环境能调用 CLI 和 GPU

In [ ]:
!which llamafactory-cli
!nvidia-smi
!python -c "import torch; print('torch:', torch.__version__); print('cuda:', torch.version.cuda); print('available:', torch.cuda.is_available())"

/root/miniconda3/bin/llamafactory-cli
Sun May 24 19:45:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.04             Driver Version: 570.124.04     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:16:00.0 Off |                  Off |
| 30%   32C    P8             27W /  450W |       1MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------

## AutoDL 4：先小样本测试训练

先用 `max_samples=100` 跑通数据读取、模型加载、LoRA 注入和 checkpoint 保存。能跑通后再全量训练。

这里只训练前100个样本

In [11]:
!llamafactory-cli train anti_fraud_qwen2_lora_sft.yaml max_samples=100

[INFO|2026-05-24 20:06:10] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:769] 2026-05-24 20:06:10,402 >> loading configuration file ../models/Qwen/Qwen2-0.5B-Instruct/config.json
[INFO|configuration_utils.py:847] 2026-05-24 20:06:10,406 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
   

## AutoDL 5：全量训练

小样本测试通过后再运行这一格。

这里yaml里面的数据文件最好数据集写绝对路径，感觉是坑点，不知道为什么微调的时候总是找不到自己的位置

In [12]:
!llamafactory-cli train anti_fraud_qwen2_lora_sft.yaml

[INFO|2026-05-24 20:07:26] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:769] 2026-05-24 20:07:26,651 >> loading configuration file ../models/Qwen/Qwen2-0.5B-Instruct/config.json
[INFO|configuration_utils.py:847] 2026-05-24 20:07:26,655 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
   